# Better preference data — controlled A/B (does mixing in Skywork-Reward make a more robust RM?)

Given the PPO ceiling (a leashed PPO hits ~59% judge and can't go higher — the **reward model** is the
bottleneck), the lever is a better RM. This tests *data diversity* with the budget held fixed:
TWO matched **Qwen2.5-0.5B-Instruct** RMs, same recipe, same **6000** pairs, differing only in mix:
- **uf**  : 6000 cleaned UltraFeedback (`train[2000:8000]`)
- **mix** : 3000 cleaned UF (`train[2000:5000]`) + 3000 Skywork-Reward-80K (`train[:3000]`)

Then evals BOTH on three held-out sets: **UF** (cleaned `train[:2000]`), **Skywork** (`train[3000:5000]`,
held out from the mix), and **HH-RLHF test** (a third distribution neither trained on = the real
generalization test). Hypothesis: mix is comparable on UF, much better on Skywork, and — the headline —
**better on HH-OOD** (diversity → robustness). ~3-4 h on a T4.

In [ ]:
import torch, json
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100=tuple(cap)==(6,0); json.dump({'p100':P100},open('/tmp/gpu.json','w'))
print(f'GPU: {name} cap={cap} -> '+('P100 fp32' if P100 else 'T4 bf16'))

In [ ]:
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)

In [ ]:
import os
!rm -rf /kaggle/working/rlhf-pipeline && git clone -q https://github.com/TheYellowDuck/RLHF-pipeline.git /kaggle/working/rlhf-pipeline
%cd /kaggle/working/rlhf-pipeline

In [ ]:
!python scripts/smoke_test.py 2>&1 | tail -2

## Config

In [ ]:
import json
P100=json.load(open('/tmp/gpu.json'))['p100']
MODEL='Qwen/Qwen2.5-0.5B-Instruct'
UF='argilla/ultrafeedback-binarized-preferences-cleaned'
SKY='Skywork/Skywork-Reward-Preference-80K-v0.2'
HH='Anthropic/hh-rlhf'
MIX=f'{UF}:train[2000:5000],{SKY}:train[:3000]'   # 3000 UF + 3000 Skywork = 6000
DTYPE,BF16=('float32','false') if P100 else ('bfloat16','true')
print('uf-only: 6000 (train[2000:8000]) | mix:', MIX)

## A) control RM — 6000 cleaned UltraFeedback only

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE \
  -o data.name=$UF -o data.train_split='train[2000:8000]' \
  -o data.max_length=512 -o train.epochs=1 -o train.batch_size=4 -o train.grad_accum=4 \
  -o train.bf16=$BF16 -o train.lr=1.0e-5 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o output_dir=checkpoints/rm_uf

## B) mix RM — 3000 cleaned UF + 3000 Skywork-Reward (same 6000 budget)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE \
  -o data.name="$MIX" -o data.train_split='train' \
  -o data.max_length=512 -o train.epochs=1 -o train.batch_size=4 -o train.grad_accum=4 \
  -o train.bf16=$BF16 -o train.lr=1.0e-5 -o train.label_smoothing=0.0 -o data.max_pair_similarity=1.0 \
  -o output_dir=checkpoints/rm_mix

## Evaluate both RMs on UF / Skywork / HH-RLHF

In [ ]:
import subprocess
def acc(ckpt, data, split):
    c=(f"python scripts/evaluate.py rm-accuracy --reward-model {ckpt} "
       f"--data '{data}' --split '{split}' --max-samples 2000 --batch-size 8")
    print('$',c,flush=True); o=subprocess.run(c,shell=True,capture_output=True,text=True)
    out=o.stdout.strip().splitlines()[-1] if o.stdout.strip() else ''
    if o.returncode: print(o.stderr[-700:])
    print(out); return out
rows={}
for tag,ck in [('uf-only','checkpoints/rm_uf'),('mix','checkpoints/rm_mix')]:
    rows[tag]=[acc(ck,UF,'train[:2000]'), acc(ck,SKY,'train[3000:5000]'), acc(ck,HH,'test')]
md=('# Better-data A/B results (Qwen2.5-0.5B-Instruct, 6000-pair budget each)\n\n'
    '| RM (train) | UF held-out | Skywork held-out | HH-RLHF (OOD, neither trained on) |\n'
    '|---|---|---|---|\n'
    f'| uf-only (6000 UF) | {rows["uf-only"][0]} | {rows["uf-only"][1]} | {rows["uf-only"][2]} |\n'
    f'| mix (3000 UF + 3000 Skywork) | {rows["mix"][0]} | {rows["mix"][1]} | {rows["mix"][2]} |\n\n'
    'Better data works if the mix is comparable on UF, much better on Skywork, and >= uf-only on HH-OOD.\n')
open('/kaggle/working/RESULTS.md','w').write(md); print('\n'+md)

### Read
The decisive cell is **HH-RLHF (OOD)**: if the mix RM generalizes better there, data diversity (not
more PPO) is the way past the ~59% ceiling — and it's worth re-running at 1.5B to build a better
reward model than the 0.8025, then PPO + judge against it.